# Filtering (stream compaction) on the GPU: chapter 12

A **filter** keeps the elements of an array that satisfy a condition and packs them at the front of the output:

This notebook goes **step by step**: 6 small programs, each adding one idea to the previous one, from the simplest kernel up. The inputs are tiny and printed, so you can follow the algorithm by hand.

The full theory, the exercises and their solutions are in `ec2/10_filter/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## Step 1 of 6: `01_sequential_filter.cu`

The sequential filter, run by a single GPU thread: keep `v < 100`, pack the kept values at the front of the output and count them.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 01_sequential_filter.cu
#include <algorithm>
#include <cstdio>
#include <iterator>
#include <vector>

#include "cuda_compat.h"

constexpr int THRESHOLD = 100;

__host__ __device__ inline bool keep(int v, int threshold) {
  return v < threshold;
}

__global__ void sequential_filter(const int* in, int* out, int* counter, int n, int threshold) {
  int kept = 0;
  for (int i = 0; i < n; ++i) {
    if (keep(in[i], threshold)) {
      out[kept] = in[i];
      ++kept;
    }
  }
  *counter = kept;
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-6s", label);
  for (int value : values) {
    std::printf("%5d", value);
  }
  std::printf("\n");
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

std::vector<int> cpuFilter(const std::vector<int>& in) {
  std::vector<int> out;
  std::copy_if(in.begin(), in.end(), std::back_inserter(out), [](int v) {
    return keep(v, THRESHOLD);
  });
  return out;
}

int main() {
  const std::vector<int> in = {
      812, 45, 30, 17, 999, 60, 431, 8, 77, 650, 230, 905, 140, 91, 3, 520};
  const int n = static_cast<int>(in.size());

  int* d_in = deviceCopyOf(in);
  int* d_out = deviceCopyOf(std::vector<int>(n, -1));
  int* d_counter = deviceCopyOf({0});

  launch(sequential_filter, dim3(1), dim3(1), d_in, d_out, d_counter, n, THRESHOLD);
  CHECK(cudaGetLastError());
  const int count = hostCopyOf(d_counter, 1)[0];
  const std::vector<int> out = hostCopyOf(d_out, count);

  std::printf("keep v < %d\n", THRESHOLD);
  printRow("in", in);
  printRow("out", out);
  std::printf("  count = %d\n", count);

  const bool pass = (out == cpuFilter(in));
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_in));
  CHECK(cudaFree(d_out));
  CHECK(cudaFree(d_counter));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 01_sequential_filter.cu -o 01_sequential_filter && ./01_sequential_filter

**What to look at in the output:** Of the 16 inputs, 8 are below 100, and `out` holds them in input order: 45 30 17 60 8 77 91 3. Only the first `count` entries of the output are meaningful. Try: set `THRESHOLD = 50`; 60, 77 and 91 drop out and the count becomes 5.

## Step 2 of 6: `02_global_atomic.cu`

One thread per element. A kept element takes its slot with `atomicAdd` on one global counter, so the output holds the right set in an unpredictable order.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 02_global_atomic.cu
#include <algorithm>
#include <cstdio>
#include <iterator>
#include <vector>

#include "cuda_compat.h"

constexpr int THRESHOLD = 100;
constexpr int BLOCK_SIZE = 4;

__host__ __device__ inline bool keep(int v, int threshold) {
  return v < threshold;
}

__global__ void filter_global_atomic(const int* in, int* out, int* counter, int n, int threshold) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    const int v = in[i];
    if (keep(v, threshold)) {
      out[atomicAdd(counter, 1)] = v;
    }
  }
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-6s", label);
  for (int value : values) {
    std::printf("%5d", value);
  }
  std::printf("\n");
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

std::vector<int> cpuFilter(const std::vector<int>& in) {
  std::vector<int> out;
  std::copy_if(in.begin(), in.end(), std::back_inserter(out), [](int v) {
    return keep(v, THRESHOLD);
  });
  return out;
}

bool sameElements(std::vector<int> a, std::vector<int> b) {
  std::sort(a.begin(), a.end());
  std::sort(b.begin(), b.end());
  return a == b;
}

int main() {
  const std::vector<int> in = {
      812, 45, 30, 17, 999, 60, 431, 8, 77, 650, 230, 905, 140, 91, 3, 520};
  const int n = static_cast<int>(in.size());
  const int blocks = static_cast<int>(cdiv(n, BLOCK_SIZE));

  int* d_in = deviceCopyOf(in);
  int* d_out = deviceCopyOf(std::vector<int>(n, -1));
  int* d_counter = deviceCopyOf({0});

  launch(
      filter_global_atomic, dim3(blocks), dim3(BLOCK_SIZE), d_in, d_out, d_counter, n, THRESHOLD);
  CHECK(cudaGetLastError());
  const int count = hostCopyOf(d_counter, 1)[0];
  const std::vector<int> out = hostCopyOf(d_out, count);
  const std::vector<int> expected = cpuFilter(in);

  std::printf("keep v < %d, %d blocks of %d threads\n", THRESHOLD, blocks, BLOCK_SIZE);
  printRow("in", in);
  printRow("out", out);
  printRow("seq", expected);
  std::printf("  same order as the sequential filter: %s\n", out == expected ? "yes" : "no");

  const bool pass = sameElements(out, expected);
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_in));
  CHECK(cudaFree(d_out));
  CHECK(cudaFree(d_counter));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 02_global_atomic.cu -o 02_global_atomic && ./02_global_atomic

**What to look at in the output:** Compare the `out` and `seq` rows: they hold the same 8 values, but in the order in which the threads reached `atomicAdd` (here 30 17 45 at the front instead of 45 30 17). Run the cell again: the order can change from run to run and may even match by chance, but the set of values never changes.

## Step 3 of 6: `03_block_private.cu`

A private counter per block in shared memory. Thread 0 reserves the block's range with one global atomic and prints it. The output is still unordered.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 03_block_private.cu
#include <algorithm>
#include <cstdio>
#include <iterator>
#include <vector>

#include "cuda_compat.h"

constexpr int THRESHOLD = 100;
constexpr int BLOCK_SIZE = 4;

__host__ __device__ inline bool keep(int v, int threshold) {
  return v < threshold;
}

__global__ void filter_block_private(const int* in, int* out, int* counter, int n, int threshold) {
  __shared__ int blockCount;
  __shared__ int blockStart;
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (threadIdx.x == 0) {
    blockCount = 0;
  }
  __syncthreads();

  const int v = (i < n) ? in[i] : 0;
  const bool k = (i < n) && keep(v, threshold);
  int local = 0;
  if (k) {
    local = atomicAdd(&blockCount, 1);
  }
  __syncthreads();

  if (threadIdx.x == 0) {
    blockStart = atomicAdd(counter, blockCount);
    printf("  block %d keeps %d, reserves out[%d..%d)\n",
           blockIdx.x,
           blockCount,
           blockStart,
           blockStart + blockCount);
  }
  __syncthreads();

  if (k) {
    out[blockStart + local] = v;
  }
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-6s", label);
  for (int value : values) {
    std::printf("%5d", value);
  }
  std::printf("\n");
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

std::vector<int> cpuFilter(const std::vector<int>& in) {
  std::vector<int> out;
  std::copy_if(in.begin(), in.end(), std::back_inserter(out), [](int v) {
    return keep(v, THRESHOLD);
  });
  return out;
}

bool sameElements(std::vector<int> a, std::vector<int> b) {
  std::sort(a.begin(), a.end());
  std::sort(b.begin(), b.end());
  return a == b;
}

int main() {
  const std::vector<int> in = {
      812, 45, 30, 17, 999, 60, 431, 8, 77, 650, 230, 905, 140, 91, 3, 520};
  const int n = static_cast<int>(in.size());
  const int blocks = static_cast<int>(cdiv(n, BLOCK_SIZE));

  int* d_in = deviceCopyOf(in);
  int* d_out = deviceCopyOf(std::vector<int>(n, -1));
  int* d_counter = deviceCopyOf({0});

  std::printf("keep v < %d, %d blocks of %d threads\n", THRESHOLD, blocks, BLOCK_SIZE);
  printRow("in", in);
  launch(
      filter_block_private, dim3(blocks), dim3(BLOCK_SIZE), d_in, d_out, d_counter, n, THRESHOLD);
  CHECK(cudaGetLastError());
  const int count = hostCopyOf(d_counter, 1)[0];
  const std::vector<int> out = hostCopyOf(d_out, count);
  const std::vector<int> expected = cpuFilter(in);

  printRow("out", out);
  printRow("seq", expected);
  std::printf("  same order as the sequential filter: %s\n", out == expected ? "yes" : "no");

  const bool pass = sameElements(out, expected);
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_in));
  CHECK(cudaFree(d_out));
  CHECK(cudaFree(d_counter));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 03_block_private.cu -o 03_block_private && ./03_block_private

**What to look at in the output:** Block 0 keeps 3 values and reserves out[0..3), block 1 reserves out[3..5), and so on: 4 global atomics in total instead of 8. Each block's values stay together in its range, but inside a range they are unordered (block 0 wrote 17 30 45), and on a GPU the blocks can also reserve their ranges in a different order.

## Step 4 of 6: `04_flags_scan_scatter.cu`

Order kept: a map kernel writes 0/1 flags, a one-block Kogge-Stone scans them, and the scatter writes `in[i]` to `out[pos[i] - 1]`. Flags and positions are printed.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 04_flags_scan_scatter.cu
#include <algorithm>
#include <cstdio>
#include <iterator>
#include <vector>

#include "cuda_compat.h"

constexpr int THRESHOLD = 100;
constexpr int SECTION_SIZE = 16;

__host__ __device__ inline bool keep(int v, int threshold) {
  return v < threshold;
}

__global__ void make_flags(const int* in, int* flags, int n, int threshold) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    flags[i] = keep(in[i], threshold) ? 1 : 0;
  }
}

__global__ void kogge_stone_scan(const int* X, int* Y, int n) {
  __shared__ int XY[SECTION_SIZE];
  const int t = threadIdx.x;
  const int i = blockIdx.x * SECTION_SIZE + t;
  XY[t] = (i < n) ? X[i] : 0;

  for (int stride = 1; stride < SECTION_SIZE; stride *= 2) {
    __syncthreads();
    int temp = XY[t];
    if (t >= stride) {
      temp += XY[t - stride];
    }
    __syncthreads();
    XY[t] = temp;
  }

  if (i < n) {
    Y[i] = XY[t];
  }
}

__global__ void scatter_kept(const int* in, const int* flags, const int* pos, int* out, int n) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n && flags[i]) {
    out[pos[i] - 1] = in[i];
  }
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-6s", label);
  for (int value : values) {
    std::printf("%5d", value);
  }
  std::printf("\n");
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

std::vector<int> cpuFilter(const std::vector<int>& in) {
  std::vector<int> out;
  std::copy_if(in.begin(), in.end(), std::back_inserter(out), [](int v) {
    return keep(v, THRESHOLD);
  });
  return out;
}

int main() {
  const std::vector<int> in = {
      812, 45, 30, 17, 999, 60, 431, 8, 77, 650, 230, 905, 140, 91, 3, 520};
  const int n = static_cast<int>(in.size());

  int* d_in = deviceCopyOf(in);
  int* d_flags = deviceCopyOf(std::vector<int>(n));
  int* d_pos = deviceCopyOf(std::vector<int>(n));
  int* d_out = deviceCopyOf(std::vector<int>(n, -1));

  launch(make_flags, dim3(1), dim3(SECTION_SIZE), d_in, d_flags, n, THRESHOLD);
  launch(kogge_stone_scan, dim3(1), dim3(SECTION_SIZE), d_flags, d_pos, n);
  launch(scatter_kept, dim3(1), dim3(SECTION_SIZE), d_in, d_flags, d_pos, d_out, n);
  CHECK(cudaGetLastError());

  const std::vector<int> pos = hostCopyOf(d_pos, n);
  const int count = pos[n - 1];
  const std::vector<int> out = hostCopyOf(d_out, count);

  std::printf("keep v < %d\n", THRESHOLD);
  printRow("in", in);
  printRow("flag", hostCopyOf(d_flags, n));
  printRow("pos", pos);
  printRow("out", out);
  std::printf("  count = pos[%d] = %d\n", n - 1, count);

  const bool pass = (out == cpuFilter(in));
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_in));
  CHECK(cudaFree(d_flags));
  CHECK(cudaFree(d_pos));
  CHECK(cudaFree(d_out));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 04_flags_scan_scatter.cu -o 04_flags_scan_scatter && ./04_flags_scan_scatter

**What to look at in the output:** `flag` marks the kept elements and `pos` is its inclusive scan, so every kept element knows its slot: 60 at index 5 has `pos` 4 and goes to out[3]. The last `pos`, 8, is the count, and `out` now has exactly the sequential order. Try: set `THRESHOLD = 50`; the flags of 60, 77 and 91 become 0, `pos` stays flat there and ends at 5.

## Step 5 of 6: `05_staged_compaction.cu`

Three 16-element sections, 4 threads with 4 elements each: count per section, scan the counts, then each block packs its kept elements in shared memory and copies them out coalesced. The last section is half padding.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 05_staged_compaction.cu
#include <algorithm>
#include <cstdio>
#include <iterator>
#include <vector>

#include "cuda_compat.h"

constexpr int THRESHOLD = 100;
constexpr int F_THREADS = 4;
constexpr int F_COARSEN = 4;
constexpr int F_SECTION = F_THREADS * F_COARSEN;

__host__ __device__ inline bool keep(int v, int threshold) {
  return v < threshold;
}

__host__ __device__ inline int exclusive_from_inclusive(const int* inclusive, int i) {
  return (i > 0) ? inclusive[i - 1] : 0;
}

__device__ void load_section_coalesced(const int* in, int n, int base, int* vals) {
  for (int k = 0; k < F_COARSEN; ++k) {
    const int j = k * F_THREADS + threadIdx.x;
    vals[j] = (base + j < n) ? in[base + j] : 0;
  }
}

__device__ bool is_real_and_kept(const int* vals, int j, int base, int n, int threshold) {
  return base + j < n && keep(vals[j], threshold);
}

__device__ void kogge_stone_in_shared(int* part) {
  const int t = threadIdx.x;
  for (int stride = 1; stride < F_THREADS; stride *= 2) {
    __syncthreads();
    int temp = part[t];
    if (t >= stride) {
      temp += part[t - stride];
    }
    __syncthreads();
    part[t] = temp;
  }
}

__device__ void copy_out_coalesced(const int* stage, int count, int* out) {
  for (int k = threadIdx.x; k < count; k += F_THREADS) {
    out[k] = stage[k];
  }
}

__device__ int block_compact(
    const int* in, int n, int base, int threshold, int* vals, int* stage, int* part) {
  const int t = threadIdx.x;
  const int first = t * F_COARSEN;

  load_section_coalesced(in, n, base, vals);
  __syncthreads();

  int count = 0;
  for (int k = 0; k < F_COARSEN; ++k) {
    if (is_real_and_kept(vals, first + k, base, n, threshold)) {
      ++count;
    }
  }
  part[t] = count;

  kogge_stone_in_shared(part);
  __syncthreads();

  int pos = exclusive_from_inclusive(part, t);
  for (int k = 0; k < F_COARSEN; ++k) {
    if (is_real_and_kept(vals, first + k, base, n, threshold)) {
      stage[pos++] = vals[first + k];
    }
  }
  __syncthreads();
  return part[F_THREADS - 1];
}

__global__ void count_kept(const int* in, int* counts, int n, int threshold) {
  __shared__ int blockCount;
  if (threadIdx.x == 0) {
    blockCount = 0;
  }
  __syncthreads();

  const int base = blockIdx.x * F_SECTION;
  int c = 0;
  for (int k = 0; k < F_COARSEN; ++k) {
    const int j = base + k * F_THREADS + threadIdx.x;
    if (j < n && keep(in[j], threshold)) {
      ++c;
    }
  }
  atomicAdd(&blockCount, c);
  __syncthreads();

  if (threadIdx.x == 0) {
    counts[blockIdx.x] = blockCount;
  }
}

__global__ void scan_counts(const int* counts, int* scannedCounts, int sections) {
  __shared__ int part[F_THREADS];
  const int t = threadIdx.x;
  part[t] = (t < sections) ? counts[t] : 0;
  kogge_stone_in_shared(part);
  if (t < sections) {
    scannedCounts[t] = part[t];
  }
}

__global__ void compact_staged(
    const int* in, int* out, const int* scannedCounts, int n, int threshold) {
  __shared__ int vals[F_SECTION];
  __shared__ int stage[F_SECTION];
  __shared__ int part[F_THREADS];
  const int count = block_compact(in, n, blockIdx.x * F_SECTION, threshold, vals, stage, part);
  const int start = exclusive_from_inclusive(scannedCounts, blockIdx.x);
  if (threadIdx.x == 0) {
    printf("  section %d: stage holds %d, goes to out[%d..%d)\n",
           blockIdx.x,
           count,
           start,
           start + count);
  }
  copy_out_coalesced(stage, count, out + start);
}

void printRows(const char* label, const std::vector<int>& values) {
  for (size_t first = 0; first < values.size(); first += F_SECTION) {
    std::printf("  %-7s", first == 0 ? label : "");
    for (size_t j = first; j < values.size() && j < first + F_SECTION; ++j) {
      std::printf("%5d", values[j]);
    }
    std::printf("\n");
  }
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

std::vector<int> cpuFilter(const std::vector<int>& in) {
  std::vector<int> out;
  std::copy_if(in.begin(), in.end(), std::back_inserter(out), [](int v) {
    return keep(v, THRESHOLD);
  });
  return out;
}

int main() {
  const std::vector<int> in = {812, 45,  30,  17,  999, 60,  431, 8,   77,  650, 230, 905, 140, 91,
                               3,   520, 5,   310, 66,  720, 12,  480, 99,  250, 870, 41,  150, 600,
                               38,  7,   940, 210, 88,  360, 515, 29,  700, 64,  130, 905};
  const int n = static_cast<int>(in.size());
  const int sections = static_cast<int>(cdiv(n, F_SECTION));

  int* d_in = deviceCopyOf(in);
  int* d_counts = deviceCopyOf(std::vector<int>(sections));
  int* d_scannedCounts = deviceCopyOf(std::vector<int>(sections));
  int* d_out = deviceCopyOf(std::vector<int>(n, -1));

  std::printf("keep v < %d, %d sections of %d elements\n", THRESHOLD, sections, F_SECTION);
  printRows("in", in);
  launch(count_kept, dim3(sections), dim3(F_THREADS), d_in, d_counts, n, THRESHOLD);
  launch(scan_counts, dim3(1), dim3(F_THREADS), d_counts, d_scannedCounts, sections);
  printRows("counts", hostCopyOf(d_counts, sections));
  printRows("scanned", hostCopyOf(d_scannedCounts, sections));
  launch(
      compact_staged, dim3(sections), dim3(F_THREADS), d_in, d_out, d_scannedCounts, n, THRESHOLD);
  CHECK(cudaGetLastError());

  const int count = hostCopyOf(d_scannedCounts, sections)[sections - 1];
  const std::vector<int> out = hostCopyOf(d_out, count);
  printRows("out", out);

  const bool countsFitOneBlock = sections <= F_THREADS;
  const bool pass = countsFitOneBlock && (out == cpuFilter(in));
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_in));
  CHECK(cudaFree(d_counts));
  CHECK(cudaFree(d_scannedCounts));
  CHECK(cudaFree(d_out));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 05_staged_compaction.cu -o 05_staged_compaction && ./05_staged_compaction

**What to look at in the output:** The counts 8 7 3 are the kept elements of each 16-element section, and their scan 8 15 18 gives each section's end in `out`, so section 1 writes out[8..15). Section 2 has only 8 real values; its 8 padding slots read as 0, which would pass `v < 100`, so `is_real_and_kept` checks the index first and the count stays 3. The 18 values come out in input order.

## Step 6 of 6: `06_single_pass.cu`

The same block compaction in one kernel: a ticket and the domino chain of `../09_scan` give each section its start, so the input is read only once.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 06_single_pass.cu
#include <algorithm>
#include <cstdio>
#include <iterator>
#include <vector>

#include "cuda_compat.h"

constexpr int THRESHOLD = 100;
constexpr int F_THREADS = 4;
constexpr int F_COARSEN = 4;
constexpr int F_SECTION = F_THREADS * F_COARSEN;
constexpr int FLAG_READY = 1;

__host__ __device__ inline bool keep(int v, int threshold) {
  return v < threshold;
}

__host__ __device__ inline int exclusive_from_inclusive(const int* inclusive, int i) {
  return (i > 0) ? inclusive[i - 1] : 0;
}

__device__ void load_section_coalesced(const int* in, int n, int base, int* vals) {
  for (int k = 0; k < F_COARSEN; ++k) {
    const int j = k * F_THREADS + threadIdx.x;
    vals[j] = (base + j < n) ? in[base + j] : 0;
  }
}

__device__ bool is_real_and_kept(const int* vals, int j, int base, int n, int threshold) {
  return base + j < n && keep(vals[j], threshold);
}

__device__ void kogge_stone_in_shared(int* part) {
  const int t = threadIdx.x;
  for (int stride = 1; stride < F_THREADS; stride *= 2) {
    __syncthreads();
    int temp = part[t];
    if (t >= stride) {
      temp += part[t - stride];
    }
    __syncthreads();
    part[t] = temp;
  }
}

__device__ void copy_out_coalesced(const int* stage, int count, int* out) {
  for (int k = threadIdx.x; k < count; k += F_THREADS) {
    out[k] = stage[k];
  }
}

__device__ int block_compact(
    const int* in, int n, int base, int threshold, int* vals, int* stage, int* part) {
  const int t = threadIdx.x;
  const int first = t * F_COARSEN;

  load_section_coalesced(in, n, base, vals);
  __syncthreads();

  int count = 0;
  for (int k = 0; k < F_COARSEN; ++k) {
    if (is_real_and_kept(vals, first + k, base, n, threshold)) {
      ++count;
    }
  }
  part[t] = count;

  kogge_stone_in_shared(part);
  __syncthreads();

  int pos = exclusive_from_inclusive(part, t);
  for (int k = 0; k < F_COARSEN; ++k) {
    if (is_real_and_kept(vals, first + k, base, n, threshold)) {
      stage[pos++] = vals[first + k];
    }
  }
  __syncthreads();
  return part[F_THREADS - 1];
}

__device__ int next_ticket(int* ticket) {
  return atomicAdd(ticket, 1);
}

__device__ int atomic_read(int* address) {
  return atomicAdd(address, 0);
}

__device__ void wait_until_ready(int* flag) {
  int state = atomic_read(flag);
  while (state != FLAG_READY) {
    state = atomic_read(flag);
  }
}

__device__ int wait_for_sum_of_previous_sections(int* flags, int* prefix, int bid) {
  if (bid == 0) {
    return 0;
  }
  wait_until_ready(&flags[bid]);
  return atomic_read(&prefix[bid]);
}

__device__ void publish_sum_of_sections_up_to(int bid, int sum, int* flags, int* prefix) {
  atomicExch(&prefix[bid + 1], sum);
  __threadfence();
  atomicExch(&flags[bid + 1], FLAG_READY);
}

__global__ void compact_single_pass(
    const int* in, int* out, int* flags, int* prefix, int* ticket, int n, int threshold) {
  __shared__ int vals[F_SECTION];
  __shared__ int stage[F_SECTION];
  __shared__ int part[F_THREADS];
  __shared__ int bid_s;
  __shared__ int start_s;

  if (threadIdx.x == 0) {
    bid_s = next_ticket(ticket);
  }
  __syncthreads();
  const int bid = bid_s;

  const int count = block_compact(in, n, bid * F_SECTION, threshold, vals, stage, part);

  if (threadIdx.x == 0) {
    const int before = wait_for_sum_of_previous_sections(flags, prefix, bid);
    publish_sum_of_sections_up_to(bid, before + count, flags, prefix);
    printf("  ticket %d: stage holds %d, sum before = %d, goes to out[%d..%d)\n",
           bid,
           count,
           before,
           before,
           before + count);
    start_s = before;
  }
  __syncthreads();

  copy_out_coalesced(stage, count, out + start_s);
}

void printRows(const char* label, const std::vector<int>& values) {
  for (size_t first = 0; first < values.size(); first += F_SECTION) {
    std::printf("  %-7s", first == 0 ? label : "");
    for (size_t j = first; j < values.size() && j < first + F_SECTION; ++j) {
      std::printf("%5d", values[j]);
    }
    std::printf("\n");
  }
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

std::vector<int> cpuFilter(const std::vector<int>& in) {
  std::vector<int> out;
  std::copy_if(in.begin(), in.end(), std::back_inserter(out), [](int v) {
    return keep(v, THRESHOLD);
  });
  return out;
}

int main() {
  const std::vector<int> in = {812, 45,  30,  17,  999, 60,  431, 8,   77,  650, 230, 905, 140, 91,
                               3,   520, 5,   310, 66,  720, 12,  480, 99,  250, 870, 41,  150, 600,
                               38,  7,   940, 210, 88,  360, 515, 29,  700, 64,  130, 905};
  const int n = static_cast<int>(in.size());
  const int sections = static_cast<int>(cdiv(n, F_SECTION));

  int* d_in = deviceCopyOf(in);
  int* d_out = deviceCopyOf(std::vector<int>(n, -1));
  int* d_flags = deviceCopyOf(std::vector<int>(sections + 1, 0));
  int* d_prefix = deviceCopyOf(std::vector<int>(sections + 1, 0));
  int* d_ticket = deviceCopyOf({0});

  std::printf("keep v < %d, %d sections of %d elements\n", THRESHOLD, sections, F_SECTION);
  printRows("in", in);
  launch(compact_single_pass,
         dim3(sections),
         dim3(F_THREADS),
         d_in,
         d_out,
         d_flags,
         d_prefix,
         d_ticket,
         n,
         THRESHOLD);
  CHECK(cudaGetLastError());

  const int count = hostCopyOf(d_prefix, sections + 1)[sections];
  const std::vector<int> out = hostCopyOf(d_out, count);
  printRows("out", out);

  const bool pass = (out == cpuFilter(in));
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_in));
  CHECK(cudaFree(d_out));
  CHECK(cudaFree(d_flags));
  CHECK(cudaFree(d_prefix));
  CHECK(cudaFree(d_ticket));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 06_single_pass.cu -o 06_single_pass && ./06_single_pass

**What to look at in the output:** The sections, ranges and `out` are the same as in step 5, but each line now comes from inside the single compaction kernel: ticket 1 waits until ticket 0 has published 8, writes its 7 values to out[8..15) and publishes 15 for ticket 2. There is no separate count kernel or scan kernel any more, so the input is read once.

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/10_filter/README.md`.

**Next:** open `colab_2_full_programs.ipynb` in this same folder.